# Machine Learning for Spatial Data — D
## Location encoding & GeoAI foundation models

Modules A–C fed models raw coordinates or a graph. But **raw `(lat, lon)` are
terrible neural-network inputs**: two nearby places can have very different
numbers, and the mapping to any target is not smooth. A **location encoder** fixes
this — a function `Enc(x) → R^d` that turns a coordinate into a smooth, multi-scale
feature vector a downstream model can use. It is the geospatial analogue of the
Transformer's positional encoding.

We build the core idea from Mai et al.'s **Space2Vec** (ICLR 2020): apply
**multi-scale sinusoids** to the coordinate, then let a small MLP consume them.
**Data:** California housing (sklearn) — it ships `Latitude`/`Longitude`, so we can
ask "how much of house value is predictable from *location alone*?"

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
torch.manual_seed(42)
np.random.seed(42)

d = fetch_california_housing(as_frame=True).frame
C = d[["Latitude", "Longitude"]].values.astype("float32")
C = (C - C.min(0)) / (C.max(0) - C.min(0))          # normalise coords to [0, 1]
y = d["MedHouseVal"].values.astype("float32")
y = (y - y.mean()) / y.std()
print(f"{len(y)} block groups; predicting house value from LOCATION ONLY")

20640 block groups; predicting house value from LOCATION ONLY


## A location encoder in ~5 lines

For each coordinate, stack `sin`/`cos` at `L` doubling frequencies (grid-cell /
Fourier features). Low frequencies capture broad trends, high frequencies capture
local detail — the "multi-scale" in Space2Vec.

In [2]:
def location_encoder(coords, L=10):
    out = []
    for i in range(L):
        f = 2.0 ** i * np.pi
        out += [np.sin(f * coords), np.cos(f * coords)]
    return np.concatenate(out, axis=1).astype("float32")

E = location_encoder(C, L=10)
print(f"raw coords dim = {C.shape[1]}   ->   encoded dim = {E.shape[1]}")

raw coords dim = 2   ->   encoded dim = 40


In [3]:
X_raw, X_enc = torch.tensor(C), torch.tensor(E)
yt = torch.tensor(y).view(-1, 1)
itr, ite = train_test_split(np.arange(len(y)), test_size=0.25, random_state=42)
tr, te = torch.tensor(itr), torch.tensor(ite)

class MLP(torch.nn.Module):
    def __init__(self, i, h=64):
        super().__init__()
        self.l1, self.l2, self.l3 = (torch.nn.Linear(i, h),
                                     torch.nn.Linear(h, h),
                                     torch.nn.Linear(h, 1))
    def forward(self, x):
        return self.l3(F.relu(self.l2(F.relu(self.l1(x)))))

def fit_score(X):
    m = MLP(X.shape[1])
    opt = torch.optim.Adam(m.parameters(), lr=0.01, weight_decay=1e-5)
    for _ in range(600):
        m.train(); opt.zero_grad()
        F.mse_loss(m(X[tr]), yt[tr]).backward(); opt.step()
    m.eval()
    with torch.no_grad():
        return r2_score(y[ite], m(X[ite]).numpy().ravel())

r2_raw = fit_score(X_raw)
r2_enc = fit_score(X_enc)
print(f"raw (lat, lon)     MLP test R^2 = {r2_raw:.3f}")
print(f"location-encoded   MLP test R^2 = {r2_enc:.3f}")

raw (lat, lon)     MLP test R^2 = 0.435
location-encoded   MLP test R^2 = 0.745


The **same** MLP, given the encoded coordinates, predicts far more of the house-
value variation from location alone — the raw-coordinate model cannot. That gap is
what a location encoder buys. (Validate spatially in production — module A.)

## The landscape: from hand-rolled encoder to foundation models

Our 5-line encoder is the seed of a fast-moving GeoAI area. Same idea, scaled up:

- **Space2Vec** (Mai et al., ICLR 2020) — multi-scale grid-cell encoder; the
  method above. https://arxiv.org/abs/2003.00824
- **Location-encoding review** (Mai et al., 2022, *IJGIS*) — the taxonomy to anchor
  the topic. https://arxiv.org/abs/2111.04006
- **SatCLIP** (Klemmer et al., AAAI 2025) — a *global* location encoder trained
  CLIP-style (match Sentinel-2 tiles to coordinates); pretrained checkpoints +
  notebooks. https://github.com/microsoft/satclip
- **GeoCLIP** — CLIP between images and GPS for worldwide geo-localisation; reuse
  its GPS encoder. `pip install geoclip`. https://github.com/VicenteVivan/geo-clip
- **PE-GNN** (Klemmer et al., AISTATS 2023) — fuses a location encoder with a GNN
  (module C) and **matches Gaussian Processes** for spatial regression, closing the
  loop back to kriging. https://github.com/konstantinklemmer/pe-gnn

**Earth-observation foundation models** (awareness — GPU/big-data, not run here):
self-supervised pretraining on massive satellite imagery, then fine-tune.

- **Prithvi-EO-2.0** (IBM/NASA) — masked-autoencoder ViT on HLS imagery;
  fine-tune for flood / burn-scar / crop mapping. https://github.com/NASA-IMPACT/Prithvi-EO-2.0
- **Presto** — a deliberately *tiny* RS foundation model, laptop-friendly feature
  extractor. https://github.com/nasaharvest/presto
- **TerraTorch** (fine-tuning toolkit) and **TorchGeo** (datasets, samplers,
  pretrained weights, CRS-aware tiling) are the libraries to use them.
  https://github.com/IBM/terratorch  ·  https://github.com/microsoft/torchgeo
- Reading list: **Awesome-Geospatial-Embeddings**.
  https://github.com/hfangcat/Awesome-Geospatial-Embeddings

**Lesson —** encode location, don't feed raw coordinates. From 5 lines of sinusoids
to billion-parameter EO models, that is the through-line of modern GeoAI.